# 04 - Attention - IMDB com GloVe

**Nome:** Ariel Ladislau Reises

Neste notebook eu treino um modelo de classificação binária para análise de sentimentos no dataset IMDB, usando embeddings pré-treinados GloVe como entrada para uma camada de auto-atenção simplificada.

A entrega contém duas versões da camada de auto-atenção:

1. **Versão com laços**: mais lenta, mas boa para entender passo a passo.
2. **Versão matricial**: sem laços na implementação da atenção, mais eficiente e adequada para minibatches.

Também deixei os embeddings congelados (`freeze=True`) para evitar overfit e seguir a orientação do exercício.

## Ferramentas e referências

Ferramentas utilizadas:

- ChatGPT: apoio para organização, revisão e comentários do notebook.
- Google Colab/Python: execução dos experimentos.

Referências usadas:

- Maas, A. L. et al. **Learning Word Vectors for Sentiment Analysis**. ACL, 2011. Dataset IMDB: https://ai.stanford.edu/~amaas/data/sentiment/
- Pennington, J.; Socher, R.; Manning, C. **GloVe: Global Vectors for Word Representation**. EMNLP, 2014. https://nlp.stanford.edu/projects/glove/
- Vaswani, A. et al. **Attention Is All You Need**. NeurIPS, 2017. https://arxiv.org/abs/1706.03762
- PyTorch. **torch.nn.Embedding**. https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html

## 1. Configurações iniciais

In [4]:
import os
from pathlib import Path

print("Diretório atual:", os.getcwd())
print("Arquivos/pastas suspeitos:")

for p in Path(".").iterdir():
    if p.name.lower() in ["torch.py", "torch", "functional.py"]:
        print(p)

Diretório atual: /content
Arquivos/pastas suspeitos:


In [5]:
import os

# Se existir torch.py, renomeia
if os.path.exists("torch.py"):
    os.rename("torch.py", "torch_local_backup.py")

# Se existir pasta torch, renomeia
if os.path.isdir("torch"):
    os.rename("torch", "torch_local_backup")

In [8]:
import sys

# limpa imports quebrados
for mod in list(sys.modules):
    if mod == "torch" or mod.startswith("torch."):
        del sys.modules[mod]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(torch.__version__)

In [1]:
!pip install --upgrade --force-reinstall torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 959.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.

In [18]:
import os
import re
import time
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Deixei tudo com seed fixa para o resultado ficar mais estável.
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Para este exercício, a entrada deve ser truncada e preenchida com padding.
params = {
    "max_length": 200,
    "glove_dim": 50,       # 50d é bem mais leve e suficiente para o exercício.
    "batch_size": 64,
    "hidden_size": 128,
    "learning_rate": 1e-3,
    "n_epochs": 5,
}

# Parametros originais:
# params = {
#     'vocabulary_size': 400000,
#     'padding_idx': 400001,
#     'max_length': 200,
# }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo usado:", device)

Dispositivo usado: cpu


## 2. Download do dataset IMDB e dos embeddings GloVe

Aqui eu baixo o dataset IMDB e o GloVe. Caso os arquivos já existam, o `wget -nc` não baixa novamente.

In [19]:
# Dataset IMDB
!wget -nc http://files.fast.ai/data/aclImdb.tgz
!tar -xzf aclImdb.tgz

# Embeddings GloVe 6B
!wget -nc http://nlp.stanford.edu/data/glove.6B.zip
!unzip -n glove.6B.zip -d glove_dir

File ‘aclImdb.tgz’ already there; not retrieving.

File ‘glove.6B.zip’ already there; not retrieving.

Archive:  glove.6B.zip


## 3. Carregamento do dataset

A base original de treino do IMDB tem 25 mil exemplos. Seguindo a instrução, eu separo **20 mil para treino** e **5 mil para validação**.

In [20]:
def load_texts(folder):
    texts = []
    for path in os.listdir(folder):
        file_path = os.path.join(folder, path)
        if os.path.isfile(file_path):
            with open(file_path, encoding="utf-8") as f:
                texts.append(f.read())
    return texts

x_train_pos = load_texts("aclImdb/train/pos")
x_train_neg = load_texts("aclImdb/train/neg")
x_test_pos = load_texts("aclImdb/test/pos")
x_test_neg = load_texts("aclImdb/test/neg")

x_all_train = x_train_pos + x_train_neg
y_all_train = [1] * len(x_train_pos) + [0] * len(x_train_neg)

x_test = x_test_pos + x_test_neg
y_test = np.array([1] * len(x_test_pos) + [0] * len(x_test_neg), dtype=np.int64)

# Embaralho antes de separar treino e validação.
combined = list(zip(x_all_train, y_all_train))
random.shuffle(combined)
x_all_train, y_all_train = zip(*combined)

x_all_train = list(x_all_train)
y_all_train = list(y_all_train)

x_train = x_all_train[:20000]
y_train = np.array(y_all_train[:20000], dtype=np.int64)

x_valid = x_all_train[20000:25000]
y_valid = np.array(y_all_train[20000:25000], dtype=np.int64)

print(len(x_train), "amostras de treino")
print(len(x_valid), "amostras de validação")
print(len(x_test), "amostras de teste")

20000 amostras de treino
5000 amostras de validação
25000 amostras de teste


## 4. Tokenização

Usei uma tokenização simples com regex para manter o foco do exercício na camada de atenção.

In [21]:
def tokenize(text):
    # Transformo tudo para minúsculo e pego sequências de letras/números.
    return [token.lower() for token in re.compile(r"\w+").findall(text)]

print(tokenize("This movie was AMAZING!!! 10/10"))

['this', 'movie', 'was', 'amazing', '10', '10']


## 5. Carregando o GloVe manualmente

Eu optei por carregar o arquivo GloVe diretamente, sem depender de `torchtext`, porque em alguns ambientes ele dá problema de instalação/compatibilidade.

Os embeddings são congelados depois no modelo.

In [22]:
def load_glove(path, dim):
    vocab = {}
    vectors = []

    with open(path, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            parts = line.rstrip().split(" ")
            word = parts[0]
            vec = np.asarray(parts[1:], dtype=np.float32)

            if vec.shape[0] != dim:
                continue

            vocab[word] = idx
            vectors.append(vec)

    vectors = torch.tensor(np.vstack(vectors), dtype=torch.float32)
    return vocab, vectors

glove_path = f"glove_dir/glove.6B.{params['glove_dim']}d.txt"
vocab, glove_vectors = load_glove(glove_path, params["glove_dim"])

# Índices extras para tokens desconhecidos e padding.
UNK_IDX = len(vocab)
PAD_IDX = len(vocab) + 1

vocab["<UNK>"] = UNK_IDX
vocab["<PAD>"] = PAD_IDX

# Vetor aleatório para desconhecido e vetor zero para padding.
torch.manual_seed(SEED)
unk_vector = torch.empty(1, params["glove_dim"]).uniform_(-0.05, 0.05)
pad_vector = torch.zeros(1, params["glove_dim"])

embeddings = torch.cat([glove_vectors, unk_vector, pad_vector], dim=0)

print("Quantidade de palavras no vocabulário:", len(vocab))
print("Shape da matriz de embeddings:", embeddings.shape)
print("UNK_IDX:", UNK_IDX)
print("PAD_IDX:", PAD_IDX)

Quantidade de palavras no vocabulário: 400002
Shape da matriz de embeddings: torch.Size([400002, 50])
UNK_IDX: 400000
PAD_IDX: 400001


## 6. Transformando texto em ids com truncamento e padding

In [23]:
def to_token_ids(text, vocab, max_length, padding_idx):
    tokens = tokenize(text)[:max_length]  # truncamento para 200 palavras
    token_ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    # Padding até max_length.
    if len(token_ids) < max_length:
        token_ids += [padding_idx] * (max_length - len(token_ids))

    return token_ids

# Teste rápido.
print(to_token_ids("this movie is great", vocab, max_length=10, padding_idx=PAD_IDX))

[37, 1005, 14, 353, 400001, 400001, 400001, 400001, 400001, 400001]


## 7. Dataset PyTorch

Aqui eu já deixo cada texto convertido para ids. Assim o treinamento fica mais rápido, porque a tokenização não é repetida a cada época.

In [24]:
class IMDBAttentionDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length, padding_idx):
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.token_ids = torch.tensor([
            to_token_ids(text, vocab, max_length, padding_idx)
            for text in texts
        ], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.token_ids[idx], self.labels[idx]

In [25]:
train_ds = IMDBAttentionDataset(x_train, y_train, vocab, params["max_length"], PAD_IDX)
valid_ds = IMDBAttentionDataset(x_valid, y_valid, vocab, params["max_length"], PAD_IDX)
test_ds = IMDBAttentionDataset(x_test, y_test, vocab, params["max_length"], PAD_IDX)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=params["batch_size"],
    shuffle=True,
    generator=generator
)

valid_loader = DataLoader(valid_ds, batch_size=params["batch_size"], shuffle=False)
test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False)

batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape, batch_y.shape)

torch.Size([64, 200]) torch.Size([64])


## 8. Auto-atenção simplificada com laços

Esta versão segue a ideia mais didática: para cada exemplo do batch, para cada token de consulta (`q`), eu comparo com todos os tokens chave (`k`), aplico softmax e faço a média ponderada dos valores.

Ela funciona, mas é lenta. Por isso, eu uso essa versão principalmente para teste e comparação.

In [26]:
class SelfAttentionLayerLoop(nn.Module):
    def __init__(self, embeddings, padding_idx):
        super().__init__()
        # Congelo os embeddings usando freeze=True.
        self.C = nn.Embedding.from_pretrained(
            embeddings.clone(),
            freeze=True,
            padding_idx=padding_idx
        )
        self.padding_idx = padding_idx

    def forward(self, batch_token_ids):
        batch_outputs = []

        # Laço no batch.
        for token_ids in batch_token_ids:
            valid_ids = token_ids[token_ids != self.padding_idx]

            # Caso raro: texto totalmente vazio após tokenização.
            if valid_ids.numel() == 0:
                batch_outputs.append(torch.zeros(self.C.embedding_dim, device=token_ids.device))
                continue

            token_contexts = []

            # Laço nas queries.
            for q_id in valid_ids:
                q = self.C(q_id)
                scores = []
                values = []

                # Laço nas keys/values.
                for k_id in valid_ids:
                    k = self.C(k_id)
                    v = self.C(k_id)

                    # Score simplificado: produto escalar entre q e k.
                    score = torch.dot(q, k)
                    scores.append(score)
                    values.append(v)

                scores = torch.stack(scores)
                values = torch.stack(values)

                weights = torch.softmax(scores, dim=0)
                context = torch.sum(weights.unsqueeze(1) * values, dim=0)
                token_contexts.append(context)

            # Depois de calcular a atenção para cada query, faço a média dos contextos.
            mean_embedding = torch.stack(token_contexts).mean(dim=0)
            batch_outputs.append(mean_embedding)

        return torch.stack(batch_outputs)

## 9. Auto-atenção simplificada matricial

Esta é a versão eficiente. A ideia é substituir os laços por operações matriciais:

- `Q = K = V = embeddings`
- `scores = QKᵀ`
- aplica máscara para ignorar padding
- `attention = softmax(scores)`
- `context = attention @ V`
- média dos contextos válidos

Na implementação matricial, não há laços dentro do `forward`.

In [27]:
class SelfAttentionLayerMatrix(nn.Module):
    def __init__(self, embeddings, padding_idx):
        super().__init__()
        self.C = nn.Embedding.from_pretrained(
            embeddings.clone(),
            freeze=True,
            padding_idx=padding_idx
        )
        self.padding_idx = padding_idx

    def forward(self, batch_token_ids):
        # E: [batch_size, seq_len, emb_dim]
        E = self.C(batch_token_ids)

        # Máscara: True para tokens válidos, False para padding.
        valid_mask = batch_token_ids != self.padding_idx

        # Scores: [batch_size, seq_len, seq_len]
        # Cada posição compara uma query com todas as keys do mesmo exemplo.
        scores = torch.matmul(E, E.transpose(1, 2))

        # Máscara das keys. Posições de padding não podem receber atenção.
        key_mask = valid_mask.unsqueeze(1)  # [batch, 1, seq_len]
        scores = scores.masked_fill(~key_mask, -1e9)

        # Pesos de atenção.
        attention_weights = torch.softmax(scores, dim=-1)

        # Contextos por posição.
        context = torch.matmul(attention_weights, E)

        # Zera contextos de queries que eram padding.
        query_mask = valid_mask.unsqueeze(-1).float()
        context = context * query_mask

        # Média apenas entre tokens válidos.
        lengths = valid_mask.sum(dim=1).clamp(min=1).unsqueeze(1).float()
        mean_embeddings = context.sum(dim=1) / lengths

        return mean_embeddings

## 10. Teste didático com embeddings falsos

Antes de treinar no IMDB, faço um teste pequeno para verificar se as duas implementações estão batendo.

In [28]:
fake_vocab = {
    "a": 0,
    "b": 1,
    "c": 2,
    "<UNK>": 3,
    "<PAD>": 4,
}

fake_embeddings = torch.arange(0, 2 * 4).reshape(4, 2).float()
fake_embeddings = torch.cat([fake_embeddings, torch.zeros(1, 2)], dim=0)

fake_examples = [
    "a",       # testa padding
    "a b",
    "a c b",   # testa truncamento
    "a z",     # testa UNK
]

fake_batch = torch.tensor([
    to_token_ids(text, fake_vocab, max_length=2, padding_idx=fake_vocab["<PAD>"])
    for text in fake_examples
], dtype=torch.long)

att_loop_fake = SelfAttentionLayerLoop(fake_embeddings, padding_idx=fake_vocab["<PAD>"])
att_matrix_fake = SelfAttentionLayerMatrix(fake_embeddings, padding_idx=fake_vocab["<PAD>"])

out_loop_fake = att_loop_fake(fake_batch)
out_matrix_fake = att_matrix_fake(fake_batch)

print("Saída com laços:")
print(out_loop_fake)
print("Saída matricial:")
print(out_matrix_fake)
print("As duas versões batem?", torch.allclose(out_loop_fake, out_matrix_fake, atol=1e-6))

assert torch.allclose(out_loop_fake, out_matrix_fake, atol=1e-6)

Saída com laços:
tensor([[0.0000, 1.0000],
        [1.8808, 2.8808],
        [3.9640, 4.9640],
        [5.9926, 6.9926]])
Saída matricial:
tensor([[0.0000, 1.0000],
        [1.8808, 2.8808],
        [3.9640, 4.9640],
        [5.9926, 6.9926]])
As duas versões batem? True


## 11. Comparação rápida de velocidade

Aqui eu comparo as duas versões em um batch pequeno. A versão com laços é propositalmente mais lenta. A versão matricial é a que deve ser usada no treino.

In [29]:
small_batch = batch_x[:8]

att_loop = SelfAttentionLayerLoop(embeddings, PAD_IDX).to(device)
att_matrix = SelfAttentionLayerMatrix(embeddings, PAD_IDX).to(device)
small_batch_device = small_batch.to(device)

start = time.time()
with torch.no_grad():
    out_loop = att_loop(small_batch_device)
loop_time = time.time() - start

start = time.time()
with torch.no_grad():
    out_matrix = att_matrix(small_batch_device)
matrix_time = time.time() - start

print("As saídas batem?", torch.allclose(out_loop, out_matrix, atol=1e-5))
print(f"Tempo com laços:   {loop_time:.4f}s")
print(f"Tempo matricial:   {matrix_time:.4f}s")

assert torch.allclose(out_loop, out_matrix, atol=1e-5)

As saídas batem? True
Tempo com laços:   3.6552s
Tempo matricial:   0.0011s


## 12. Modelo de classificação

O modelo usa:

1. Embeddings GloVe congelados;
2. Camada de auto-atenção simplificada;
3. Camada linear escondida;
4. Camada de saída para classificação binária.

In [30]:
class AttentionClassifier(nn.Module):
    def __init__(self, embeddings, padding_idx, hidden_size, output_size=2, attention_type="matrix"):
        super().__init__()

        if attention_type == "matrix":
            self.attention = SelfAttentionLayerMatrix(embeddings, padding_idx)
        elif attention_type == "loop":
            self.attention = SelfAttentionLayerLoop(embeddings, padding_idx)
        else:
            raise ValueError("attention_type deve ser 'matrix' ou 'loop'.")

        input_size = embeddings.shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, token_ids):
        x = self.attention(token_ids)
        logits = self.classifier(x)
        return logits

## 13. Funções de treino e avaliação

In [31]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            preds = logits.argmax(dim=1)

            total_loss += loss.item() * x_batch.size(0)
            total_correct += (preds == y_batch).sum().item()
            total_examples += x_batch.size(0)

    return total_loss / total_examples, total_correct / total_examples


def train_model(model, train_loader, valid_loader, params, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

    best_valid_acc = 0.0
    best_state = None
    history = []

    for epoch in range(params["n_epochs"]):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_examples = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * x_batch.size(0)
            total_correct += (preds == y_batch).sum().item()
            total_examples += x_batch.size(0)

        train_loss = total_loss / total_examples
        train_acc = total_correct / total_examples
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "valid_loss": valid_loss,
            "valid_acc": valid_acc,
        })

        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Época {epoch+1:02d}/{params['n_epochs']} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"valid_loss={valid_loss:.4f} | valid_acc={valid_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

## 14. Treinamento do modelo matricial

Para o treino principal, uso a implementação matricial, porque ela é muito mais eficiente. A versão com laços já foi entregue e testada, mas seria inviável treinar tudo com ela em tempo razoável.

In [32]:
torch.manual_seed(SEED)

model_matrix = AttentionClassifier(
    embeddings=embeddings,
    padding_idx=PAD_IDX,
    hidden_size=params["hidden_size"],
    output_size=2,
    attention_type="matrix"
).to(device)

print(model_matrix)

model_matrix, history_matrix = train_model(
    model_matrix,
    train_loader,
    valid_loader,
    params,
    device
)

AttentionClassifier(
  (attention): SelfAttentionLayerMatrix(
    (C): Embedding(400002, 50, padding_idx=400001)
  )
  (classifier): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)
Época 01/5 | train_loss=0.6426 | train_acc=0.6312 | valid_loss=0.6086 | valid_acc=0.6660
Época 02/5 | train_loss=0.5816 | train_acc=0.6978 | valid_loss=0.5704 | valid_acc=0.7054
Época 03/5 | train_loss=0.5645 | train_acc=0.7099 | valid_loss=0.5603 | valid_acc=0.7116
Época 04/5 | train_loss=0.5577 | train_acc=0.7149 | valid_loss=0.5737 | valid_acc=0.6954
Época 05/5 | train_loss=0.5501 | train_acc=0.7229 | valid_loss=0.5543 | valid_acc=0.7208


## 15. Avaliação final no teste

In [33]:
criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(model_matrix, test_loader, criterion, device)

print("Resultado final no teste")
print(f"Loss: {test_loss:.4f}")
print(f"Acurácia: {test_acc:.4f}")

Resultado final no teste
Loss: 0.5588
Acurácia: 0.7107


## 16. Observação sobre a versão com laços

A implementação com laços está correta e foi comparada com a matricial. Porém, para treinar com 20 mil amostras e sequências de 200 tokens, ela fica muito lenta, porque calcula a atenção token por token.

A versão matricial faz a mesma operação usando multiplicações de matrizes e máscaras, por isso é a implementação adequada para o treinamento real.

## 17. Conclusão

Neste exercício, implementei uma camada de auto-atenção simplificada de duas formas: uma com laços, para entender melhor o funcionamento interno, e outra matricial, para ganhar eficiência no treinamento com minibatches.

Usei embeddings GloVe pré-treinados como entrada e mantive esses embeddings congelados, conforme solicitado. O modelo foi treinado no dataset IMDB com 20 mil amostras de treino e 5 mil de validação, e a acurácia final foi medida no conjunto de teste.

A principal diferença prática entre as duas implementações é o desempenho: a versão com laços é útil para aprendizado, mas a matricial é muito mais rápida e adequada para uso real.